# GameTheory 04e — Oracles réflexifs, décision causale et équilibre de Nash

> **Source primaire** : Benja Fallenstein, Jessica Taylor, Paul F. Christiano (2015), *Reflective Oracles: A Foundation for Classical Game Theory*. arXiv : [1508.04145](https://arxiv.org/abs/1508.04145). Bibliothèque canonique privée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\2015 - Fallenstein Taylor Christiano - Reflective Oracles - A Foundation for Classical Game Theory.pdf` (SHA-256 `c0ae7668...ee6653`).

Ce notebook étend la grappe `GameTheory-04*` en introduisant l'**auto-référence computationnelle** : un agent qui peut se demander *ce qu'un autre agent, capable de le modéliser, ferait*. Le fil conducteur est qu'une boucle naïve (Matching Pennies avec deux agents déterministes qui se prédisent) **mène à une contradiction diagonale** ; l'article propose une **restriction finie** du problème (queries, fermeture, borne) qui le rend illustrable numériquement. L'objectif pédagogique est de rendre l'articulation entre :

- théorie causale de la décision (CDT) et équilibre de Nash (théorème 4.1),
- la construction d'un oracle réflexif *fini* (théorème 5.1),
- les limites : le résultat fini illustratif **n'est pas** l'oracle universel du théorème 2.1 (point fixe de Kakutani en dimension infinie).

**Position dans le parcours Expert / informatique théorique** : prolonge `GameTheory-04-NashEquilibrium` (stratégies mixtes, Matching Pennies) et `GameTheory-04c-NashExistence-Python` (point fixe Brouwer discriminant — cellule `perturbed_br`, anti-tautologie Prong-B). Le 04c montre qu'**un point fixe peut être rendu discriminant** ; le 04e montre qu'**un agent qui se modélise lui-même peut être rendu fini et illustrable**, sans confondre cette restriction avec l'oracle universel.

**Acceptance à venir (multi-cycle)** :
1. Boucle / Matching Pennies : deux agents déterministes qui se prédisent mènent à une contradiction (sous-section 1).
2. Définition d'une *requête réflexive* `(M, p)` et des deux implications strictes (sous-section 2).
3. *Menteur probabiliste* `M^O() = 1 − O(M, 1/2)` : la solution cohérente randomise à `1/2` (sous-section 3).
4. Théorème 3.1 : encoder une comparaison d'utilités comme requête d'oracle ; vérifier que l'action choisie maximise l'utilité espérée selon CDT (sous-section 4).
5. Théorème 4.1 : relier les probabilités d'action d'agents intégrés à un équilibre de Nash (sous-section 5).
6. Restriction *finie, fermée et bornée* inspirée du théorème 5.1, construire son jeu auxiliaire, résoudre avec Nashpy, vérifier les contraintes de réflexivité (sous-section 6).
7. *Frontière de l'implémentation* : indiquer sans ambiguïté que le théorème d'existence général (théorème 2.1, point fixe de Kakutani) n'est **pas** implémenté par la démo finie.


## 0. Imports et constantes

L'environnement n'a aucune dépendance exotique : `numpy` pour les vecteurs de probabilités, `nashpy` pour la résolution d'équilibre, et les outils Python standard. Les seeds sont fixées pour la reproductibilité — un test discriminant n'est crédible que s'il bat la même graine à chaque exécution.


In [1]:
import numpy as np
from typing import Callable, Tuple

# Reproductibilité — un test discriminant doit battre la même graine à chaque exécution.
# Si une expérience "passe" à seed=0 mais "échoue" à seed=42, ce n'est pas un test.
SEED = 0
rng = np.random.default_rng(SEED)

# Bornes par défaut — voir sous-section 6 (théorème 5.1) pour la justification.
N_QUERIES_MAX = 64  # borne sur le nombre d'appels à l'oracle dans la restriction finie
TOL = 1e-9          # tolérance pour les égalités de probabilité (CDT vs simulée)

print("GameTheory-04e — squelette charge (kernel python3).")


GameTheory-04e — squelette charge (kernel python3).


## 1. Boucle et contradiction — Matching Pennies avec deux agents déterministes

Soit deux agents L (Ligne) et C (Colonne) qui jouent à Matching Pennies. Si chacun est **déterministe** et **sait quel algorithme joue l'autre**, alors l'un peut déduire la sortie de l'autre et choisir l'action qui gagne à coup sûr. Mais l'autre peut faire le même raisonnement. La seule issue stable est une boucle — pas un point fixe.

Ce que cette section **doit montrer** : pour toute paire de stratégies pures `(s_L, s_C)` dans `{0,1}²`, on a `gain(s_L, s_C)` ≠ équilibre de Nash. Donc Matching Pennies **n'a pas** d'équilibre en stratégies pures. La section suivante relâche la contrainte "déterministe" en autorisant le mélange.


In [2]:
# Placeholder de discrimination Prong-B : la section 1 doit iterer sur les 4 paires
# de strategies pures et montrer l'absence d'equilibre (sinon la discrimination est
# tautologique comme `perturbed_br` a (0.5,0.5) avant PR #7664).
def pure_strategy_payoff(sL: int, sC: int) -> Tuple[int, int]:
    """Matching Pennies : Ligne gagne si sL == sC (Pile/Face identique).

    Matrice canonique : Ligne cherche l'egalite, Colonne cherche la difference.
    Renvoie (gain_Ligne, gain_Colonne). Equilibre de Nash mixte attendu a (1/2, 1/2),
    mais pas d'equilibre pur : c'est ce qu'il faut demontrer ici.
    """
    # TODO: implementer la matrice et le calcul du payoff ; montrer pour les 4 paires
    # l'absence d'equilibre (L ne peut pas faire mieux unilateralement ET C non plus).
    pass

# Refus de tautologie : la demonstration d'"aucun equilibre pur" doit balayer les 4 paires
# (00, 01, 10, 11) et verifier best-response mutuelle -- sans quoi la cellule est une
# affirmation sans preuve. Voir anti-tautologie Prong-B (`perturbed_br` #7664).


## 2. Requête réflexive `(M, p)`

Une *requête réflexive* est un couple `(M, p)` où `M` est une **machine** (au sens de la calculabilité — par exemple une fonction `M : {0,1}* → {0,1}`) et `p ∈ [0,1]` une probabilité. La *réponse* à la requête est le résultat de l'exécution de `M` sur sa propre représentation et la requête `p` — notation `M(M, p)` quand `M` accepte une requête probabiliste.

L'article distingue deux implications strictes :

- **Cohérence** : si `M(M, p)` produit `b ∈ {0,1}` avec probabilité 1, alors `b = M(M, p)` exécuté *à nouveau* produit `b` avec probabilité 1 — pas de dépendance sur un état caché.
- **Réflexivité stricte** : `M` qui prétend "simuler un autre agent `N`" doit retourner ce que `N` retournerait sur la même entrée, modulo la requête probabiliste `p`.

Ces deux conditions sont nécessaires pour parler *d'oracle réflexif* — et la section 3 montre que sans une restriction finie, on tombe sur le menteur probabiliste qui n'a pas de solution close.


In [3]:
# Stubs pour la cellule d'introduction -- ne pas implementer `M` avant la section 6
# (la restriction finie). Ici on documente le TYPE attendu.
class ReflexiveQuery:
    """Stub pour (M, p). M doit etre appelable par elle-meme -- la coherence est
    triviale si M est purement fonctionnelle mais PI si M maintient un etat.
    """
    def __init__(self, M: Callable, p: float):
        # TODO: validation reflexive -- coherence et reflexivite stricte (sections 2 et 5).
        pass

    def answer(self) -> int:
        # TODO: placeholder jusqu'a la section 6 (restriction finie + jeux auxiliaires).
        pass


## 3. Le menteur probabiliste

Considérons l'opérateur `M^O()` défini par `M^O() = 1 − O(M^O, 1/2)` où `O` est un oracle capable d'évaluer `M^O` avec une requête probabiliste `1/2`.

Si `M^O` retourne `b ∈ {0,1}` avec probabilité 1, alors la définition impose :

- soit `b = 0` : alors `O(M^O, 1/2) = 0`, donc `1 − 0 = 1` — contradiction.
- soit `b = 1` : alors `O(M^O, 1/2) = 1`, donc `1 − 1 = 0` — contradiction.

Aucune solution déterministe. Mais si on autorise `M^O` à *randomiser*, alors la solution cohérente est `M^O` qui retourne `1` avec probabilité `1/2` (et `0` avec probabilité `1/2`). À l'équilibre, `O(M^O, 1/2) = 1/2`, et `1 − 1/2 = 1/2` — cohérent.

**Ce que cette section doit démontrer numériquement** : sur un échantillon de suffisamment de tirages, la moyenne empirique tend vers `1/2` (avec un écart-type `√(p(1−p)/n)` cohérent). Si la moyenne dévie, c'est un *biais d'implémentation* — pas une réfutation.


In [4]:
def probabilistic_liar() -> int:
    """M^O() = 1 - O(M^O, 1/2). Solution coherente : randomiser a 1/2.

    Cette cellule est un *stub* : l'implementation necessite l'oracle O, qui n'existe
    pas avant la section 6 (restriction finie). Le stub retourne un tirage uniforme
    pour *illustrer le theoreme* -- il ne le prouve pas.
    """
    # TODO: remplacer par un appel a un oracle reflexif fini (section 6).
    return int(rng.integers(0, 2))  # ILLUSTRATION SEULE -- pas une preuve

# Verification : la moyenne empirique doit tendre vers 1/2.
# (A executer apres la cellule 6 -- pas de preuve sans l'oracle reel.)


## 4. Théorème 3.1 — Décision causale et oracle utilitaire

Le théorème 3.1 de l'article énonce qu'on peut encoder une comparaison d'utilités `U(a) > U(b)` comme une requête d'oracle `(M_U, p)` où `M_U` est une machine qui implémente la théorie causale de la décision. L'action choisie par l'agent maximisera l'utilité espérée.

**Section à venir** (Cycle 2+) : démonstration numérique sur un problème-jouet (par exemple Stag Hunt, où CDT et EDT diffèrent) avec un oracle réflexif *fini* sur les valeurs d'utilité.


## 5. Théorème 4.1 — Agents intégrés et équilibre de Nash

Si les agents d'un jeu sont *intégrés* — c'est-à-dire si leurs probabilités d'action sont les réponses à des requêtes d'oracle cohérentes — alors le profil agrégé est un équilibre de Nash du jeu sous-jacent.

**Section à venir** (Cycle 2+) : démontrer numériquement sur un jeu 2×2 (par exemple Stag Hunt) que le profil induit par les oracles finis est un équilibre de Nash au sens *best-response mutuelle* — chaque agent maximise unilatéralement étant donné l'autre.


## 6. Restriction finie (théorème 5.1) et jeu auxiliaire

Le théorème 5.1 donne une **restriction finie, fermée et bornée** des requêtes autorisées :

- *Finie* : un ensemble fini de paires `(M, p)` est autorisé (par exemple `N_QUERIES_MAX`).
- *Fermée* : toute réponse d'un oracle dans la restriction est elle-même une requête autorisée.
- *Bornée* : la profondeur d'imbrication des requêtes est bornée (au plus une profondeur `D`).

Avec ces trois conditions, l'ensemble des profils d'actions possibles pour les agents du jeu est fini. On construit un **jeu auxiliaire** où chaque joueur est une fonction des réponses autorisées, et on résout l'équilibre avec Nashpy. La vérification de réflexivité compare les probabilités prédites par le jeu auxiliaire aux réponses effectives des oracles.

**Section à venir** (Cycle 2+) : implémentation sur un jeu 2×2 avec `N_QUERIES_MAX = 64` et `D = 2`. Le vérificateur indépendant des contraintes de réflexivité est essentiel — sans lui, on a une affirmation sans preuve.


## 7. Frontière de l'implémentation

**Ce que ce notebook n'est PAS** :

- Une implémentation de l'oracle universel du théorème 2.1 (point fixe de Kakutani en dimension infinie).
- Une prétention à la calculabilité générale, à l'oracle de l'arrêt, ou à la résolution de tous les jeux par auto-référence.
- Un nouveau notebook Lean : la formalisation des oracles réflexifs reste *future work* (à cadrer par un EPIC séparé).

**Ce que ce notebook est** : une illustration *finie* d'un mécanisme que l'article développe en général. Le théorème 5.1 prouve l'existence d'une restriction finie utile ; ce notebook la construit sur un cas-jouet et la vérifie numériquement. Le théorème 2.1, lui, requiert des outils de convexité en dimension infinie (Kakutani) qui sortent du scope d'un notebook Python pédagogique.

Si cette frontière n'est pas explicite, le lecteur risque d'inférer que la démo finie tient lieu d'oracle universel — c'est précisément la confusion que l'article travaille à dissiper.


## 8. Exercices

Les exercices ci-dessous sont conformes à la règle C.1 (stubs sans `raise NotImplementedError` — un notebook doit s'exécuter de bout en bout même exercices non complétés).

### Exercice 1 — Boucle et meilleure réponse pure

Compléter `pure_strategy_payoff(sL, sC)` dans la section 1 pour qu'elle retourne le couple `(gain_L, gain_C)` selon la matrice canonique de Matching Pennies. Ensuite, montrer sur les 4 paires `(0,0), (0,1), (1,0), (1,1)` qu'aucune n'est un équilibre de Nash pur (L peut faire mieux unilatéralement OU C, et dans les deux cas l'autre joueur doit pouvoir le faire aussi pour parler d'équilibre).

**Indice** : pour Matching Pennies, Ligne gagne si les deux jouaient la même face. Donc la matrice `gain_L` est `[[1, 0], [0, 1]]` (sL == sC → 1, sinon 0). Le gain de Colonne est l'inverse.

Critère d'acceptation : la cellule imprime les 4 paires avec les gains, marque `Aucune paire en équilibre pur` et l'écrit explicitement. Un test qui se contente d'imprimer les gains **manque** la démonstration.


In [5]:
# Exercice 1 -- verification : la definition de pure_strategy_payoff vit en
# section 1 et n'est pas redefinie ici. Ce bloc appelle la fonction sur les
# 4 paires et affiche le verdict attendu.
for sL in (0, 1):
    for sC in (0, 1):
        gL, gC = pure_strategy_payoff(sL, sC) or (None, None)
        # TODO etudiant : imprimer les gains et verifier best-response mutuelle.
        print(f"L={sL} C={sC}  gain_L={gL} gain_C={gC}  -- TODO verifier")

print("Exercice 1 a completer : aucun equilibre pur attendu.")

L=0 C=0  gain_L=None gain_C=None  -- TODO verifier
L=0 C=1  gain_L=None gain_C=None  -- TODO verifier
L=1 C=0  gain_L=None gain_C=None  -- TODO verifier
L=1 C=1  gain_L=None gain_C=None  -- TODO verifier
Exercice 1 a completer : aucun equilibre pur attendu.


### Exercice 2 — Menteur probabiliste : cohérence de la moyenne empirique

Compléter la cellule de la section 3 pour vérifier que la moyenne empirique de `probabilistic_liar()` sur `n = 10 000` tirages est dans l'intervalle `[1/2 − 3σ, 1/2 + 3σ]` où `σ = √(1/(4n)) ≈ 0.005`. Si elle est hors de cet intervalle sur 10 000 tirages, c'est un biais d'implémentation (graine, méthode, etc.) — pas une réfutation.

**Indice** : intervalle `0.5 ± 3 * np.sqrt(0.25 / n)`. Si la moyenne tombe dedans, l'illustration numérique est cohérente avec la théorie.

Critère d'acceptation : la cellule imprime la moyenne empirique et l'intervalle de confiance, et conclut `OK` ou `BIAIS` selon le cas.


In [6]:
# Exercice 2 -- squelette a completer
n = 10_000
samples = np.array([probabilistic_liar() for _ in range(n)])
mean = samples.mean()
se = np.sqrt(0.25 / n)
# TODO etudiant : comparer mean a l'intervalle [0.5 - 3*se, 0.5 + 3*se] et conclure OK ou BIAIS.
print(f"moyenne empirique={mean:.4f}  SE={se:.4f}  intervalle=[{0.5-3*se:.4f}, {0.5+3*se:.4f}]")
print("Exercice 2 a completer : verifier la coherence numerique.")


moyenne empirique=0.5030  SE=0.0050  intervalle=[0.4850, 0.5150]


Exercice 2 a completer : verifier la coherence numerique.


### Exercice 3 — Restriction finie : choisir le bon horizon

Pour un jeu 2×2 et une profondeur `D = 2`, combien de **profils** `(s_L, s_C)` distincts la restriction finie autorise-t-elle au plus (en supposant que chaque agent peut randomiser sur `K` sorties à chaque profondeur) ?

**Indice** : à profondeur 1, chaque agent a `K` profils ; à profondeur 2, chaque agent a `K` profils de profondeur 1 × `K` profils internes = `K²` profils totaux. Pour Matching Pennies (`K = 2` stratégies pures), c'est `4` profils par agent à profondeur 2.

Critère d'acceptation : la cellule imprime le nombre total de profils et le compare au nombre de paires pures (`4`) et mixtes (`∞` en continu).


In [7]:
# Exercice 3 -- reponse par calcul direct
K = 2     # Matching Pennies : Pile/Face
D = 2     # profondeur d'imbrication autorisee
n_profils = K ** D   # profils par agent a profondeur D
# TODO etudiant : commenter ce que ca implique pour la cardinalite de la restriction finie.
print(f"K={K}, D={D}  -> {n_profils} profils par agent, {n_profils**2} paires totales.")
print("Exercice 3 a completer : discuter la cardinalite vs Nash continu.")


K=2, D=2  -> 4 profils par agent, 16 paires totales.
Exercice 3 a completer : discuter la cardinalite vs Nash continu.


## 9. Conclusion

Ce notebook ouvre la porte aux **oracles réflexifs** dans la grappe `GameTheory-04*`. Le squelette actuel (Cycle 1) installe les fondations conceptuelles, les exercices squelettes, et la *frontière de l'implémentation*. Les sections 1-3 sont esquissées numériquement, les sections 4-6 attendent les implémentations effectives des oracles finis. Le notebook complet reste un livrable multi-cycles — voir l'issue #14450 pour le détail des acceptance criteria.

**Repères amont** : `GameTheory-04` (Nash, Lemke-Howson) · `GameTheory-04c-Python` (Brouwer discriminant, anti-tautologie Prong-B). **Repères aval visés** (à venir) : `DecInfer-1-Utility-Foundations` pour les liens CDT, `GameTheory-11-BayesianGames` pour les jeux à information incomplète.

**Frontière maintenue** : ce notebook est une illustration *finie*, pas une implémentation de l'oracle universel. Toute lecture qui confondrait les deux trahit l'intention de l'article.
